# 数据清单与原始检查

按功能合并的报告：先给主要结果，附录保留早期实验与复核记录。各章节的数据范围、阈值版本和输出彼此独立。

运行某一章节时，先重启 kernel，再从该章节的初始化单元开始；人工交互章节按其中的说明操作。运行对应的 builder 会更新该章节，其他章节保持不变。

---

## 主要结果：bigp3bci_raw_inspection

# StudyF 原始 EEG 检查：一个被试、一个 session

本 notebook 只做 raw data inspection，不进行滤波、ICA、伪迹修复、epoch 切分或 target/non-target 解码。

当前范围：

- 被试：`F_03`
- Session：`SE001`
- 数据：`Train/CBCol` 下的 EDF 文件

bigP3BCI 使用 EDF 格式，因此使用 `mne.io.read_raw_edf()`，而不是教程示例中的 `read_raw_fif()`。


In [ ]:
%matplotlib qt
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne

PROJECT_ROOT=Path.cwd()
while PROJECT_ROOT!=PROJECT_ROOT.parent and not ((PROJECT_ROOT/'README.md').exists() or (PROJECT_ROOT/'.git').exists()):
    PROJECT_ROOT=PROJECT_ROOT.parent
if not ((PROJECT_ROOT/'README.md').exists() or (PROJECT_ROOT/'.git').exists()):
    raise FileNotFoundError('请在项目目录或子目录启动 Jupyter。')

SUBJECT_ID='F_03'
SESSION_ID='SE001'
TRAIN_ROOT=PROJECT_ROOT/'data'/'raw'/'bigP3BCI-data'/'StudyF'/SUBJECT_ID/SESSION_ID/'Train'/'CBCol'
if not TRAIN_ROOT.is_dir():
    raise FileNotFoundError(f'找不到：{TRAIN_ROOT}')

train_files=sorted(TRAIN_ROOT.glob('*.edf'))
if not train_files:
    raise FileNotFoundError(f'目录中没有 EDF 文件：{TRAIN_ROOT}')

raw_train={path.name:mne.io.read_raw_edf(path,preload=False,infer_types=True,verbose=False) for path in train_files}

def assign_bigp3bci_channel_types(raw):
    import re
    event_names={'StimulusBegin','StimulusType','StimulusCode','CurrentTarget','FakeFeedback','DisplayResults','DisplayResult','SelectedTarget','SelectedRow','SelectedColumn','PhaseInSequence'}
    channel_types={}
    for name in raw.ch_names:
        if name in event_names:
            channel_types[name]='stim'
        elif re.match(r'^.+_\d+_\d+$',name):
            channel_types[name]='misc'
        elif name.upper().startswith('EOG'):
            channel_types[name]='eog'
    if channel_types:
        raw.set_channel_types(channel_types,on_unit_change='ignore',verbose=False)
    return raw

raw_train={name:assign_bigp3bci_channel_types(raw) for name,raw in raw_train.items()}
print('MNE:',mne.__version__)
print('Subject:',SUBJECT_ID)
print('Session:',SESSION_ID)
print('EDF files:',len(raw_train))
print([path.name for path in train_files])
print('Corrected channel types:', dict(zip(next(iter(raw_train.values())).ch_names, next(iter(raw_train.values())).get_channel_types())))


In [ ]:
# 明确修正 bigP3BCI EDF 的通道类型
import re

EVENT_CHANNELS = {
    'StimulusBegin', 'StimulusType', 'StimulusCode', 'CurrentTarget',
    'FakeFeedback', 'DisplayResults', 'DisplayResult', 'SelectedTarget',
    'SelectedRow', 'SelectedColumn', 'PhaseInSequence'
}

for filename, raw in raw_train.items():
    type_map = {}
    for name in raw.ch_names:
        if name in EVENT_CHANNELS:
            type_map[name] = 'stim'
        elif re.match(r'^.+_\d+_\d+$', name):
            type_map[name] = 'misc'
        elif name.upper().startswith('EOG'):
            type_map[name] = 'eog'
    raw.set_channel_types(type_map, on_unit_change='ignore', verbose=False)

channel_type_check = pd.concat([
    pd.DataFrame({
        'file': filename,
        'channel': raw.ch_names,
        'type': raw.get_channel_types(),
    })
    for filename, raw in raw_train.items()
], ignore_index=True)

display(channel_type_check)
print('Stim/event types:', channel_type_check[channel_type_check['channel'].isin(EVENT_CHANNELS)]['type'].value_counts().to_dict())
print('EEG count by file:')
display(channel_type_check.groupby(['file', 'type']).size().unstack(fill_value=0))


## 1. 查看数据概况

先确认每个 EDF 都能正常读取，并记录：

- MNE 的 `Raw` 摘要；
- `raw.info`；
- 通道名称和数量；
- 采样率；
- 记录时长；
- 已标记的 bad channels；
- 通道类型。


In [ ]:
for filename, raw in raw_train.items():
    print('\n' + '=' * 90)
    print(filename)
    print(raw)
    print('\nraw.info:')
    print(raw.info)
    print('\nNumber of channels:', len(raw.ch_names))
    print('Sampling frequency:', raw.info['sfreq'], 'Hz')
    print('Duration:', raw.times[-1], 'seconds')
    print('Bad channels:', raw.info['bads'])
    print('Channel types:')
    display(pd.DataFrame({'channel':raw.ch_names,'type':raw.get_channel_types()}))


## 2. 画一小段连续 EEG 信号

按照原始数据检查的要求，这里只画约 20 秒的 EEG 通道。暂时不滤波、不重参考，也不做 DC correction；先观察是否存在：

- 平直通道；
- 突然的大幅跳变；
- 明显高频噪声；
- 某个通道长期远高于其他通道；
- 明显低频漂移。

字符呈现通道、Stimulus/event 通道和反馈通道不参与 EEG 波形检查。


In [ ]:
def eeg_channels_only(raw):
    return [name for name, kind in zip(raw.ch_names, raw.get_channel_types()) if kind == 'eeg' and name not in raw.info['bads']]

for filename, raw in raw_train.items():
    eeg=eeg_channels_only(raw)
    print(filename, 'EEG channels:', len(eeg))
    if eeg:
        raw.plot(duration=min(20,raw.times[-1]),start=0,picks=eeg,n_channels=len(eeg),remove_dc=False,scalings='auto',title=f'{filename}: raw EEG, first 20 seconds',show=True)


## 3. 查看未滤波频谱

这里直接对原始 EEG 计算 PSD，不先滤波。重点观察：

- 能量主要分布在哪些频段；
- 低频能量是否异常突出；
- 50/60 Hz 附近是否有明显电源线噪声峰；
- 不同 EEG 通道的频谱是否明显异常。

此步骤只用于检查，不在这里应用 notch 或 band-pass filter。


In [ ]:
for filename, raw in raw_train.items():
    eeg=eeg_channels_only(raw)
    if not eeg:
        continue
    spectrum=raw.compute_psd(fmin=0.1,fmax=min(250,raw.info['sfreq']/2-1),picks=eeg,reject_by_annotation=False,exclude='bads',verbose=False)
    spectrum.plot(average=True,amplitude=False,show=True)
    print(filename, ': unfiltered PSD displayed')


## 4. 查看 annotations 和原始事件信息

先查看 `raw.annotations` 的数量、时间和 description。这里不把 annotation 自动转换成事件，也不解码 `StimulusBegin`、`StimulusType` 或 `StimulusCode`。

同时列出可能的 trigger/event/status/target 通道名称，后续再单独核对它们的编码定义。


In [ ]:
EVENT_WORDS=('stimulus','event','trigger','status','marker','target','feedback','display','phase','selected')

for filename, raw in raw_train.items():
    print('\n' + '=' * 90)
    print(filename)
    print('Number of annotations:', len(raw.annotations))

    if len(raw.annotations):
        annotation_table=pd.DataFrame({'onset':raw.annotations.onset,'duration':raw.annotations.duration,'description':raw.annotations.description})
        display(annotation_table)
    else:
        print('No annotations found.')

    possible_channels=[name for name in raw.ch_names if any(word in name.lower() for word in EVENT_WORDS)]
    print('Possible trigger/event channels:', possible_channels or 'none found')
    print('All non-EEG channels:')
    display(pd.DataFrame({'channel':raw.ch_names,'type':raw.get_channel_types()}).query("type != 'eeg'"))


## 5. 保留原始数据不变

当前 notebook 中的 `raw_train` 使用 `preload=False` 读取，且没有对原始对象执行滤波、裁剪、重参考、坏道插值或 ICA。

如果后续需要尝试裁剪、选通道或滤波，必须先复制：


In [ ]:
# 示例：只对副本进行探索，不改变 raw_train 中的原始 Raw
example_filename=next(iter(raw_train))
raw_original=raw_train[example_filename]
raw_copy=raw_original.copy()

# 后续探索应在 raw_copy 上进行，例如：
# raw_copy.crop(tmax=60)
# raw_copy.pick(eeg_channels_only(raw_copy))
# raw_copy.filter(l_freq=0.1, h_freq=40.0)

print('Original duration:', raw_original.times[-1])
print('Copy duration:', raw_copy.times[-1])
print('Original object remains unchanged.')


## 当前阶段结论记录

完成上述图形和表格检查后，再记录：

- 文件是否均能读取；
- 通道数量和采样率是否一致；
- 是否存在已有 bad channels；
- 是否存在明显低频漂移；
- 是否存在 50/60 Hz 峰；
- annotations 是否为空；
- 哪些通道可能承担 trigger/event/status 作用。

本 notebook 到这里结束，不在本阶段解释事件编码或生成 P300 标签。


## 6. 导出完整通道表

下面不截断显示所有通道，并将通道清单导出为 CSV。表中保留文件名、通道序号、通道名称、MNE 识别类型、采样率、记录时长以及 raw.info 中的原始采集滤波元数据。


In [ ]:
channel_rows=[]
for filename, raw in raw_train.items():
    for index, (name, kind) in enumerate(zip(raw.ch_names, raw.get_channel_types())):
        channel_rows.append({
            'file': filename,
            'channel_index': index,
            'channel_name': name,
            'mne_type': kind,
            'sfreq_hz': raw.info['sfreq'],
            'duration_sec': raw.times[-1],
            'raw_highpass_hz': raw.info['highpass'],
            'raw_lowpass_hz': raw.info['lowpass'],
            'is_bad': name in raw.info['bads'],
        })

channel_inventory = pd.DataFrame(channel_rows)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

display(channel_inventory)

export_dir = PROJECT_ROOT / 'outputs' / 'raw_inspection'
export_dir.mkdir(parents=True, exist_ok=True)
export_path = export_dir / f'{SUBJECT_ID}_{SESSION_ID}_channel_inventory.csv'
channel_inventory.to_csv(export_path, index=False, encoding='utf-8-sig')
print(f'完整通道表已导出：{export_path}')


## 7. 通道类型与用途简表

下面按通道名称去重，只保留一行，列出 MNE 类型和在 bigP3BCI 中的简易用途说明。


## 8. Stimulus/event 通道审计

本节不预先假设所有通道都有效，而是逐个检查它们的取值、非零比例、唯一值和 stimulus onset 对齐值。

保留策略：

- 核心标签：`StimulusBegin`、`StimulusType`、`CurrentTarget`
- 刺激顺序和阶段：`StimulusCode`、`PhaseInSequence`
- Test 反馈结果：`SelectedTarget`、`SelectedRow`、`SelectedColumn`、`DisplayResults`、`FakeFeedback`
- 全零通道不用于建模，但保留在审计表中记录。


In [ ]:
STIMULUS_CHANNELS = [
    'StimulusBegin', 'StimulusType', 'StimulusCode', 'CurrentTarget',
    'PhaseInSequence', 'SelectedTarget', 'SelectedRow', 'SelectedColumn',
    'DisplayResults', 'DisplayResult', 'FakeFeedback',
]

CORE_LABEL_CHANNELS = ['StimulusBegin', 'StimulusType', 'CurrentTarget']
SEQUENCE_CHANNELS = ['StimulusCode', 'PhaseInSequence']
FEEDBACK_CHANNELS = ['SelectedTarget', 'SelectedRow', 'SelectedColumn', 'DisplayResults', 'DisplayResult', 'FakeFeedback']

channel_audit_rows = []
onset_event_tables = []

for filename, raw in raw_train.items():
    present = [name for name in STIMULUS_CHANNELS if name in raw.ch_names]

    onset_samples = np.array([], dtype=int)
    if 'StimulusBegin' in raw.ch_names:
        begin = raw.get_data(picks=['StimulusBegin'], verbose=False)[0]
        onset_samples = np.flatnonzero(
            np.diff((begin > 0.5).astype(int), prepend=0) == 1
        )

    for name in present:
        values = raw.get_data(picks=[name], verbose=False)[0]
        unique_values, counts = np.unique(values, return_counts=True)
        nonzero = values != 0
        transitions = np.count_nonzero(np.diff(values) != 0)

        channel_audit_rows.append({
            'file': filename,
            'channel': name,
            'role': (
                'core label' if name in CORE_LABEL_CHANNELS
                else 'sequence/phase' if name in SEQUENCE_CHANNELS
                else 'feedback/result' if name in FEEDBACK_CHANNELS
                else 'other'
            ),
            'min': values.min(),
            'max': values.max(),
            'std': values.std(),
            'nonzero_count': int(nonzero.sum()),
            'nonzero_fraction': float(nonzero.mean()),
            'unique_count': len(unique_values),
            'unique_values_first': unique_values[:20].tolist(),
            'transition_count': int(transitions),
            'all_zero': bool(np.all(values == 0)),
            'values_at_onset_first': values[onset_samples[:10]].tolist() if len(onset_samples) else [],
        })

    if len(onset_samples):
        onset_table = pd.DataFrame({
            'file': filename,
            'sample': onset_samples,
            'time_sec': onset_samples / raw.info['sfreq'],
        })
        for name in present:
            values = raw.get_data(picks=[name], verbose=False)[0]
            onset_table[name] = values[onset_samples]
        onset_event_tables.append(onset_table)

channel_audit = pd.DataFrame(channel_audit_rows)
onset_events = pd.concat(onset_event_tables, ignore_index=True) if onset_event_tables else pd.DataFrame()

print('Channel audit:')
display(channel_audit)

print('All-zero channels by file:')
display(channel_audit[channel_audit['all_zero']][['file', 'channel', 'role']])

print('Values at first stimulus onsets:')
display(onset_events.head(20))

# Export both tables
export_dir = PROJECT_ROOT / 'outputs' / 'raw_inspection'
export_dir.mkdir(parents=True, exist_ok=True)
channel_audit.to_csv(export_dir / f'{SUBJECT_ID}_{SESSION_ID}_stimulus_channel_audit.csv', index=False, encoding='utf-8-sig')
onset_events.to_csv(export_dir / f'{SUBJECT_ID}_{SESSION_ID}_stimulus_onset_values.csv', index=False, encoding='utf-8-sig')
print('Audit tables exported to:', export_dir)


## 9. 全组 Train/CBCol 的 EEG 数据统计

本节统计研究范围内所有被试和 Session 的 `Train/CBCol/*.edf` 文件。

这里只纳入 MNE 通道类型为 `eeg` 的通道。`stim`、`misc`、`eog` 等通道不会进入 EEG 数值统计，因此 `StimulusBegin`、`StimulusType`、`StimulusCode`、`CurrentTarget` 等事件通道不会污染下面的均值、标准差和幅值范围。

每个 EDF 文件单独读取和统计，避免把不同 Session 的数据直接拼接。

In [ ]:
GROUP_TRAIN_ROOT = PROJECT_ROOT / 'data' / 'raw' / 'bigP3BCI-data' / 'StudyF'
GROUP_TRAIN_FILES = sorted(GROUP_TRAIN_ROOT.glob('*/SE*/Train/CBCol/*.edf'))

if not GROUP_TRAIN_FILES:
    raise FileNotFoundError(f'No Train/CBCol EDF files found under: {GROUP_TRAIN_ROOT}')

group_file_rows = []
group_channel_rows = []

for path in GROUP_TRAIN_FILES:
    raw = mne.io.read_raw_edf(path, preload=False, infer_types=True, verbose=False)
    raw = assign_bigp3bci_channel_types(raw)
    eeg_names = [name for name, kind in zip(raw.ch_names, raw.get_channel_types()) if kind == 'eeg']

    if not eeg_names:
        print('WARNING: no EEG channels:', path.name)
        continue

    # MNE returns EEG data in volts; convert only the copied statistics to microvolts.
    eeg_data_uV = raw.get_data(picks=eeg_names, verbose=False) * 1e6
    subject = path.parts[-5]
    session = path.parts[-4]

    group_file_rows.append({
        'subject': subject,
        'session': session,
        'file': path.name,
        'n_eeg_channels': len(eeg_names),
        'sfreq_hz': raw.info['sfreq'],
        'duration_sec': raw.times[-1],
        'eeg_mean_uV': float(eeg_data_uV.mean()),
        'eeg_std_uV': float(eeg_data_uV.std()),
        'eeg_min_uV': float(eeg_data_uV.min()),
        'eeg_max_uV': float(eeg_data_uV.max()),
        'n_flat_eeg_channels': int(np.sum(np.ptp(eeg_data_uV, axis=1) == 0)),
    })

    for name, values in zip(eeg_names, eeg_data_uV):
        group_channel_rows.append({
            'subject': subject,
            'session': session,
            'file': path.name,
            'channel': name,
            'mne_type': 'eeg',
            'sfreq_hz': raw.info['sfreq'],
            'mean_uV': float(values.mean()),
            'std_uV': float(values.std()),
            'min_uV': float(values.min()),
            'max_uV': float(values.max()),
            'p01_uV': float(np.percentile(values, 1)),
            'p99_uV': float(np.percentile(values, 99)),
            'flat': bool(np.ptp(values) == 0),
        })

group_file_stats = pd.DataFrame(group_file_rows)
group_eeg_stats = pd.DataFrame(group_channel_rows)

print(f'Files included: {len(group_file_stats)}')
print(f'EEG channel records: {len(group_eeg_stats)}')
print(f'Unique EEG channel names: {group_eeg_stats["channel"].nunique()}')
print("All rows above have mne_type == 'eeg'.")

display(group_file_stats)
display(group_eeg_stats.head(20))

print('EEG channel counts by file:')
display(group_file_stats.groupby(['subject', 'session'])['n_eeg_channels'].agg(['count', 'min', 'max']))

print('Flat EEG channels:')
display(group_eeg_stats[group_eeg_stats['flat']])

group_export_dir = PROJECT_ROOT / 'outputs' / 'raw_inspection'
group_export_dir.mkdir(parents=True, exist_ok=True)
group_file_stats.to_csv(group_export_dir / 'StudyF_Train_CBCol_EEG_file_statistics.csv', index=False, encoding='utf-8-sig')
group_eeg_stats.to_csv(group_export_dir / 'StudyF_Train_CBCol_EEG_channel_statistics.csv', index=False, encoding='utf-8-sig')
print('Saved:', group_export_dir / 'StudyF_Train_CBCol_EEG_file_statistics.csv')
print('Saved:', group_export_dir / 'StudyF_Train_CBCol_EEG_channel_statistics.csv')

## 10. 全组 EEG 频谱：不同 Subject 与总体平均

下面沿用前面的 PSD 图，但将所有 `Train/CBCol` 文件汇总。不同颜色表示不同 Subject，黑色粗线表示全组平均频谱。

计算时只选择 MNE 类型为 `eeg` 的通道，不包含任何 `stim` 或 `misc` 通道。每个文件先对其 EEG 通道求平均，再在文件层面求平均，避免记录较长的文件对总体结果产生过大影响。

In [ ]:
subject_session_psd = {}
all_file_psd = []
psd_freqs = None

for path in GROUP_TRAIN_FILES:
    raw = mne.io.read_raw_edf(path, preload=False, infer_types=True, verbose=False)
    raw = assign_bigp3bci_channel_types(raw)
    eeg_names = [name for name, kind in zip(raw.ch_names, raw.get_channel_types()) if kind == 'eeg']
    if not eeg_names:
        continue

    spectrum = raw.compute_psd(
        fmin=0.1,
        fmax=min(125, raw.info['sfreq'] / 2 - 1),
        picks=eeg_names,
        reject_by_annotation=False,
        exclude='bads',
        verbose=False,
    )
    freqs = spectrum.freqs
    power = spectrum.get_data().mean(axis=0)
    power_db_uV = 10 * np.log10(np.maximum(power * 1e12, np.finfo(float).tiny))

    if psd_freqs is None:
        psd_freqs = freqs
    subject = path.parts[-5]
    session = path.parts[-4]
    subject_session_psd.setdefault((subject, session), []).append(power_db_uV)
    all_file_psd.append(power_db_uV)

subjects = sorted({subject for subject, session in subject_session_psd})
sessions = sorted({session for subject, session in subject_session_psd})
subject_colors = dict(zip(subjects, plt.cm.tab20(np.linspace(0, 1, max(len(subjects), 1)))))
fig, axes = plt.subplots(1, len(sessions), figsize=(7 * len(sessions), 5), sharey=True, constrained_layout=True)
axes = np.atleast_1d(axes)
for ax, session in zip(axes, sessions):
    session_spectra = []
    for subject in subjects:
        spectra = subject_session_psd.get((subject, session), [])
        if not spectra:
            continue
        subject_mean = np.mean(spectra, axis=0)
        session_spectra.extend(spectra)
        ax.plot(psd_freqs, subject_mean, color=subject_colors[subject], linewidth=1.2, alpha=0.8, label=subject)

    if session_spectra:
        session_grand_mean = np.mean(session_spectra, axis=0)
        ax.plot(psd_freqs, session_grand_mean, color='black', linewidth=3, label=f'{session} average')
    ax.axvline(60, color='gray', linestyle='--', linewidth=1, alpha=0.8)
    ax.set_xlim(0.1, 125)
    ax.set_xlabel('Frequency (Hz)')
    ax.set_title(session)
    ax.grid(alpha=0.3)
axes[0].set_ylabel('Power (dB/Hz, µV²)')
fig.suptitle('StudyF Train/CBCol: EEG PSD by Session', fontsize=14)
from matplotlib.lines import Line2D
subject_handles = [Line2D([0], [0], color=subject_colors[s], lw=2, label=s) for s in subjects]
subject_handles.append(Line2D([0], [0], color='black', lw=3, label='Session average'))
axes[-1].legend(handles=subject_handles, title='Subject / color', loc='upper right', fontsize=8)
plt.show()

print('Files included:', len(all_file_psd))
print('Subject-session combinations included:', sorted(subject_session_psd))

## 11. Duke 与 ETSU 两个采集机构的 EEG PSD 对比

本图按照当前的推断映射分为两个采集机构：Duke（`F_05`–`F_08`）和 ETSU（`F_03`、`F_20`、`F_21`、`F_23`、`F_24`、`F_25`）。行表示机构，列表示 Session。每条彩色曲线代表一个 Subject，黑色粗线表示该机构在该 Session 内的平均 PSD。

In [ ]:
duke_subjects = {'F_05', 'F_06', 'F_07', 'F_08'}
etsu_subjects = {'F_03', 'F_20', 'F_21', 'F_23', 'F_24', 'F_25'}
available_subjects = {subject for subject, session in subject_session_psd}
group_subjects = {
    'Duke (inferred)': sorted(available_subjects & duke_subjects),
    'ETSU (inferred)': sorted(available_subjects & etsu_subjects),
}

fig, axes = plt.subplots(2, len(sessions), figsize=(7 * len(sessions), 9), sharex=True, sharey=True, constrained_layout=True)
axes = np.asarray(axes)
subject_colors = dict(zip(sorted(available_subjects), plt.cm.tab20(np.linspace(0, 1, max(len(available_subjects), 1)))))

for row, (group_name, subjects_in_group) in enumerate(group_subjects.items()):
    for col, session in enumerate(sessions):
        ax = axes[row, col]
        session_spectra = []
        for subject in subjects_in_group:
            spectra = subject_session_psd.get((subject, session), [])
            if not spectra:
                continue
            subject_mean = np.mean(spectra, axis=0)
            session_spectra.extend(spectra)
            ax.plot(psd_freqs, subject_mean, color=subject_colors[subject], linewidth=1.2, alpha=0.8, label=subject)

        if session_spectra:
            group_mean = np.mean(session_spectra, axis=0)
            ax.plot(psd_freqs, group_mean, color='black', linewidth=3, label='Group average')
        ax.axvline(60, color='gray', linestyle='--', linewidth=1, alpha=0.7)
        ax.set_xlim(0.1, 125)
        ax.grid(alpha=0.3)
        if row == 0:
            ax.set_title(session)
        if col == 0:
            ax.set_ylabel(f'{group_name}\nPower (dB/Hz, µV²)')
        if row == 1:
            ax.set_xlabel('Frequency (Hz)')
        ax.legend(fontsize=8)

fig.suptitle('StudyF Train/CBCol: EEG PSD comparison of Subject groups', fontsize=14)
plt.show()

print('Duke subjects included:', group_subjects['Duke (inferred)'])
print('ETSU subjects included:', group_subjects['ETSU (inferred)'])

## 12. 审计每个 Train/Test 文件中的实际字符数量

本节遍历 StudyF 的所有 EDF，分别统计 `Train/CBCol`、`Test/Static`、`Test/Dyn` 和 `Test/DynBigram`。

字符数不使用 `CurrentTarget` 的 unique 数量，因为同一个目标字符可能在不同字符位置重复出现。这里使用刺激时刻的 `CurrentTarget` 连续目标区块估计字符边界，同时报告 flash 数和 `StimulusType` 标签数量，便于核对。

In [ ]:
ALL_STUDY_F_FILES = sorted((PROJECT_ROOT / 'data' / 'raw' / 'bigP3BCI-data' / 'StudyF').rglob('*.edf'))

def rising_onsets(values, threshold=0.5):
    binary = np.asarray(values) > threshold
    return np.flatnonzero(binary & ~np.r_[False, binary[:-1]])

def count_nonzero_runs(values):
    values = np.asarray(values)
    nonzero = values != 0
    return int(np.sum(nonzero & ~np.r_[False, nonzero[:-1]]))

character_audit_rows = []
for edf_path in ALL_STUDY_F_FILES:
    raw_header = mne.io.read_raw_edf(edf_path, preload=False, infer_types=True, verbose='ERROR')
    available = set(raw_header.ch_names)
    required = {'StimulusBegin', 'CurrentTarget'}
    if not required.issubset(available):
        print('Skipped: missing required event channels:', edf_path.name)
        continue

    picks = ['StimulusBegin', 'CurrentTarget']
    if 'StimulusType' in available:
        picks.append('StimulusType')
    if 'SelectedTarget' in available:
        picks.append('SelectedTarget')
    event_data = raw_header.get_data(picks=picks, verbose=False)
    values = dict(zip(picks, event_data))
    flash_onsets = rising_onsets(values['StimulusBegin'])
    target_at_flash = values['CurrentTarget'][flash_onsets]
    valid_target = target_at_flash != 0
    target_block_starts = valid_target & ~np.r_[False, valid_target[:-1]]
    target_changes = valid_target & np.r_[True, target_at_flash[1:] != target_at_flash[:-1]] if len(target_at_flash) else np.array([], dtype=bool)

    selected_target_events = (
        count_nonzero_runs(values['SelectedTarget'])
        if 'SelectedTarget' in values else np.nan
    )
    character_count = (
        selected_target_events
        if np.isfinite(selected_target_events) and selected_target_events > 0
        else int(target_changes.sum())
    )

    row = {
        'subject': edf_path.parts[-5],
        'session': edf_path.parts[-4],
        'split': edf_path.parts[-3],
        'condition': edf_path.parts[-2],
        'file': edf_path.name,
        'n_flash_onsets': len(flash_onsets),
        'n_current_target_nonzero_blocks': int(target_block_starts.sum()),
        'n_current_target_changes_at_flash': int(target_changes.sum()),
        'n_unique_current_target_values_at_flash': int(np.unique(target_at_flash[target_at_flash != 0]).size),
        'current_target_values_at_flash': np.unique(target_at_flash[target_at_flash != 0]).tolist(),
        'estimated_characters': int(character_count),
        'sfreq_hz': raw_header.info['sfreq'],
    }
    if 'StimulusType' in values:
        row['n_target_flashes'] = int(np.sum(values['StimulusType'][flash_onsets] == 1))
        row['n_non_target_flashes'] = int(np.sum(values['StimulusType'][flash_onsets] == 0))
    else:
        row['n_target_flashes'] = np.nan
        row['n_non_target_flashes'] = np.nan
    row['n_selected_target_events'] = selected_target_events
    character_audit_rows.append(row)

character_audit = pd.DataFrame(character_audit_rows)
display(character_audit)

print('Summary by subject, session, split and condition:')
summary = (
    character_audit
    .groupby(['subject', 'session', 'split', 'condition'], as_index=False)
    .agg(
        n_files=('file', 'count'),
        estimated_characters_min=('estimated_characters', 'min'),
        estimated_characters_max=('estimated_characters', 'max'),
        total_flash_onsets=('n_flash_onsets', 'sum'),
    )
)
display(summary)

character_audit_export = PROJECT_ROOT / 'outputs' / 'raw_inspection' / 'StudyF_all_conditions_character_audit.csv'
character_summary_export = PROJECT_ROOT / 'outputs' / 'raw_inspection' / 'StudyF_all_conditions_character_summary.csv'
character_audit.to_csv(character_audit_export, index=False, encoding='utf-8-sig')
summary.to_csv(character_summary_export, index=False, encoding='utf-8-sig')
print('Saved:', character_audit_export)
print('Saved:', character_summary_export)

## 13. 审计 FakeFeedback 与 DisplayResults 通道

本节只读取两个状态通道，不加载 EEG 数据。逐个 EDF 检查它们是否全为 0，或者只在某些 Train/Test 条件中有实际记录。

In [ ]:
FEEDBACK_CHANNELS = ['FakeFeedback', 'DisplayResults']
feedback_rows = []

for edf_path in ALL_STUDY_F_FILES:
    raw_header = mne.io.read_raw_edf(edf_path, preload=False, infer_types=True, verbose='ERROR')
    present = [name for name in FEEDBACK_CHANNELS if name in raw_header.ch_names]
    if not present:
        feedback_rows.append({
            'subject': edf_path.parts[-5],
            'session': edf_path.parts[-4],
            'split': edf_path.parts[-3],
            'condition': edf_path.parts[-2],
            'file': edf_path.name,
            'channel': None,
            'present': False,
        })
        continue

    data = raw_header.get_data(picks=present, verbose=False)
    for channel, values in zip(present, data):
        unique_values, counts = np.unique(values, return_counts=True)
        feedback_rows.append({
            'subject': edf_path.parts[-5],
            'session': edf_path.parts[-4],
            'split': edf_path.parts[-3],
            'condition': edf_path.parts[-2],
            'file': edf_path.name,
            'channel': channel,
            'present': True,
            'min': float(values.min()),
            'max': float(values.max()),
            'std': float(values.std()),
            'nonzero_count': int(np.count_nonzero(values)),
            'unique_count': int(len(unique_values)),
            'unique_values': unique_values.tolist(),
            'all_zero': bool(np.all(values == 0)),
        })

feedback_audit = pd.DataFrame(feedback_rows)
display(feedback_audit)

print('Summary by split and condition:')
feedback_summary = (
    feedback_audit[feedback_audit['present'] == True]
    .groupby(['split', 'condition', 'channel'], as_index=False)
    .agg(
        n_files=('file', 'count'),
        n_all_zero=('all_zero', 'sum'),
        max_value=('max', 'max'),
        total_nonzero_samples=('nonzero_count', 'sum'),
    )
)
display(feedback_summary)

feedback_export = PROJECT_ROOT / 'outputs' / 'raw_inspection' / 'StudyF_feedback_display_audit.csv'
feedback_summary_export = PROJECT_ROOT / 'outputs' / 'raw_inspection' / 'StudyF_feedback_display_summary.csv'
feedback_audit.to_csv(feedback_export, index=False, encoding='utf-8-sig')
feedback_summary.to_csv(feedback_summary_export, index=False, encoding='utf-8-sig')
print('Saved:', feedback_export)
print('Saved:', feedback_summary_export)

## 14. 单个 6-character Train 文件的 EEG 与事件核对

本节暂不滤波、不切 epoch、不改变原始数据。先核对 EEG 单位与参考信息，并检查 `StimulusBegin`、`StimulusType`、`CurrentTarget`、字符状态通道之间的关系。

In [ ]:
AUDIT_FILE = PROJECT_ROOT / 'data' / 'raw' / 'bigP3BCI-data' / 'StudyF' / 'F_03' / 'SE001' / 'Train' / 'CBCol' / 'F_03_SE001_CBCol_Train01.edf'
audit_raw = mne.io.read_raw_edf(AUDIT_FILE, preload=False, infer_types=True, verbose='ERROR')
audit_raw = assign_bigp3bci_channel_types(audit_raw)

EEG_NAMES = [name for name in audit_raw.ch_names if name.upper().startswith('EEG_')]
CHARACTER_NAMES = [name for name in audit_raw.ch_names if re.match(r'^.+_\d+_\d+$', name)]
EVENT_NAMES = ['StimulusBegin', 'StimulusType', 'StimulusCode', 'CurrentTarget', 'SelectedTarget']
EVENT_NAMES = [name for name in EVENT_NAMES if name in audit_raw.ch_names]
PICKS_FOR_AUDIT = EEG_NAMES + CHARACTER_NAMES + EVENT_NAMES
audit_data = audit_raw.get_data(picks=PICKS_FOR_AUDIT, verbose=False)
audit_values = dict(zip(PICKS_FOR_AUDIT, audit_data))

print('File:', AUDIT_FILE.name)
print('Sampling frequency:', audit_raw.info['sfreq'], 'Hz')
print('Duration:', audit_raw.times[-1], 's')
print('EEG channels:', len(EEG_NAMES))

unit_rows = []
for name in EEG_NAMES:
    idx = audit_raw.ch_names.index(name)
    ch_info = audit_raw.info['chs'][idx]
    unit_rows.append({
        'channel': name,
        'mne_unit_code': ch_info['unit'],
        'unit_multiplier': ch_info['unit_mul'],
        'raw_data_unit': 'V (MNE convention)',
        'display_unit': 'µV in MNE plots',
    })
print('EEG unit metadata:')
display(pd.DataFrame(unit_rows))
print('custom_ref_applied:', audit_raw.info.get('custom_ref_applied'))
print('montage:', audit_raw.get_montage())
print('description:', audit_raw.info.get('description'))
print('subject_info:', audit_raw.info.get('subject_info'))

def rising_edges(values, threshold=0.5):
    binary = np.asarray(values) > threshold
    return np.flatnonzero(binary & ~np.r_[False, binary[:-1]])

stimulus_onsets = rising_edges(audit_values['StimulusBegin'])
stimulus_begin_changes = np.flatnonzero(np.diff(audit_values['StimulusBegin']) != 0) + 1
print('StimulusBegin rising edges:', len(stimulus_onsets))
print('StimulusBegin all value changes:', len(stimulus_begin_changes))
print('First stimulus onset samples:', stimulus_onsets[:10].tolist())

onset_table = pd.DataFrame({
    'sample': stimulus_onsets,
    'time_sec': stimulus_onsets / audit_raw.info['sfreq'],
    'stimulus_code': audit_values['StimulusCode'][stimulus_onsets] if 'StimulusCode' in audit_values else np.nan,
    'stimulus_type': audit_values['StimulusType'][stimulus_onsets] if 'StimulusType' in audit_values else np.nan,
    'current_target': audit_values['CurrentTarget'][stimulus_onsets] if 'CurrentTarget' in audit_values else np.nan,
})
print('StimulusType values at flash onsets:', sorted(onset_table['stimulus_type'].dropna().unique().tolist()))
display(onset_table.head(20))

selected_target_onsets = rising_edges(audit_values['SelectedTarget']) if 'SelectedTarget' in audit_values else np.array([], dtype=int)
print('SelectedTarget events:', len(selected_target_onsets))

if len(selected_target_onsets):
    character_rows = []
    previous_boundary = -1
    for character_index, boundary in enumerate(selected_target_onsets, start=1):
        flash_mask = (stimulus_onsets > previous_boundary) & (stimulus_onsets < boundary)
        flash_indices = np.flatnonzero(flash_mask)
        target_values = onset_table.loc[flash_indices, 'stimulus_type'].to_numpy()
        current_values = onset_table.loc[flash_indices, 'current_target'].to_numpy()
        character_rows.append({
            'character_trial': character_index,
            'n_flashes': len(flash_indices),
            'n_target_epochs': int(np.sum(target_values == 1)),
            'n_non_target_epochs': int(np.sum(target_values == 0)),
            'target_label_values': sorted(np.unique(current_values).tolist()),
            'missing_stimulus_type': bool(np.any(~np.isin(target_values, [0, 1]))),
        })
        previous_boundary = boundary
    character_table = pd.DataFrame(character_rows)
    print('Character-trial audit:')
    display(character_table)
else:
    character_table = pd.DataFrame()
    print('No SelectedTarget events found; character boundaries require a separate rule.')

active_character_rows = []
for sample in stimulus_onsets:
    active = [name for name in CHARACTER_NAMES if audit_values[name][sample] > 0]
    active_character_rows.append({
        'sample': sample,
        'n_active_character_channels': len(active),
        'active_character_channels': active,
    })
active_character_table = pd.DataFrame(active_character_rows)
print('Character-channel activity at first flash onsets:')
display(active_character_table.head(20))

print('Missing stimulus labels:', int(onset_table['stimulus_type'].isna().sum()))
print('Invalid StimulusType values:', int((~onset_table['stimulus_type'].isin([0, 1])).sum()))
print('Flash-onset sample duplicates:', int(onset_table['sample'].duplicated().sum()))
print('StimulusCode duplicate values at adjacent flashes:', int(np.sum(np.diff(onset_table['stimulus_code']) == 0)))


## 15. 审计 StimulusCode 的取值范围

分别统计 `StimulusCode` 在整个通道中的非零取值范围，以及它在 `StimulusBegin` flash 上升沿时的取值范围。

In [ ]:
stimulus_code_rows = []
for edf_path in ALL_STUDY_F_FILES:
    raw_header = mne.io.read_raw_edf(edf_path, preload=False, infer_types=True, verbose='ERROR')
    if not {'StimulusCode', 'StimulusBegin'}.issubset(raw_header.ch_names):
        continue
    values = raw_header.get_data(picks=['StimulusCode', 'StimulusBegin'], verbose=False)
    stimulus_code = values[0]
    stimulus_begin = values[1]
    nonzero_code = stimulus_code[stimulus_code != 0]
    onset_samples = rising_onsets(stimulus_begin)
    onset_code = stimulus_code[onset_samples]
    onset_nonzero_code = onset_code[onset_code != 0]
    stimulus_code_rows.append({
        'subject': edf_path.parts[-5],
        'session': edf_path.parts[-4],
        'split': edf_path.parts[-3],
        'condition': edf_path.parts[-2],
        'file': edf_path.name,
        'overall_min_nonzero': float(nonzero_code.min()) if len(nonzero_code) else np.nan,
        'overall_max_nonzero': float(nonzero_code.max()) if len(nonzero_code) else np.nan,
        'overall_unique_nonzero': int(np.unique(nonzero_code).size),
        'onset_min_nonzero': float(onset_nonzero_code.min()) if len(onset_nonzero_code) else np.nan,
        'onset_max_nonzero': float(onset_nonzero_code.max()) if len(onset_nonzero_code) else np.nan,
        'onset_unique_nonzero': int(np.unique(onset_nonzero_code).size),
        'n_onsets': int(len(onset_samples)),
        'n_zero_code_at_onset': int(np.sum(onset_code == 0)),
    })

stimulus_code_audit = pd.DataFrame(stimulus_code_rows)
display(stimulus_code_audit)
print('Overall range by condition:')
display(stimulus_code_audit.groupby(['split', 'condition'])[['overall_min_nonzero', 'overall_max_nonzero', 'overall_unique_nonzero']].agg(['min', 'max']))
print('Range at StimulusBegin onsets by condition:')
display(stimulus_code_audit.groupby(['split', 'condition'])[['onset_min_nonzero', 'onset_max_nonzero', 'onset_unique_nonzero', 'n_zero_code_at_onset']].agg(['min', 'max']))

stimulus_code_export = PROJECT_ROOT / 'outputs' / 'raw_inspection' / 'StudyF_StimulusCode_range_audit.csv'
stimulus_code_audit.to_csv(stimulus_code_export, index=False, encoding='utf-8-sig')
print('Saved:', stimulus_code_export)

## 16. 固定的逐 character event-QC

完整 character 的标准为 126 flashes、14 target、112 non-target、CurrentTarget 恒定、StimulusType 只有 0/1、StimulusCode 连续或合法重置、相邻 flash 间隔约为 0.25 秒。Train/CBCol 和 Test/Static 使用固定标准；Dyn 条件单独标记。

In [ ]:
EXPECTED_FLASHES, EXPECTED_TARGET, EXPECTED_NON_TARGET = 126, 14, 112
EXPECTED_INTERVAL_SEC = 0.25
character_qc_rows, file_qc_rows = [], []

def classify_code_sequence(codes):
    codes = np.asarray(codes)
    if len(codes) == 0 or np.any(codes == 0):
        return 'invalid_or_zero'
    if len(codes) == 1 or np.all(np.diff(codes) == 1):
        return 'monotonic_within_character'
    return 'non_monotonic_or_unexpected'

for edf_path in ALL_STUDY_F_FILES:
    raw_header = mne.io.read_raw_edf(edf_path, preload=False, infer_types=True, verbose='ERROR')
    required = ['StimulusBegin', 'StimulusType', 'StimulusCode', 'CurrentTarget', 'SelectedTarget']
    missing = [name for name in required if name not in raw_header.ch_names]
    base = {'subject': edf_path.parts[-5], 'session': edf_path.parts[-4], 'split': edf_path.parts[-3], 'condition': edf_path.parts[-2], 'file': edf_path.name}
    if missing:
        file_qc_rows.append({**base, 'n_character_trials': 0, 'file_status': 'needs_review', 'review_notes': f'missing channels: {missing}'})
        continue
    data = raw_header.get_data(picks=required, verbose=False)
    values = dict(zip(required, data))
    flash_onsets = rising_onsets(values['StimulusBegin'])
    character_boundaries = rising_onsets(values['SelectedTarget'])
    rows_for_file = []
    previous_boundary = -1
    fixed_standard = (edf_path.parts[-3] == 'Train' and edf_path.parts[-2] == 'CBCol') or (edf_path.parts[-3] == 'Test' and edf_path.parts[-2] == 'Static')
    for character_index, boundary in enumerate(character_boundaries, start=1):
        mask = (flash_onsets > previous_boundary) & (flash_onsets < boundary)
        samples = flash_onsets[mask]
        stimulus_type = values['StimulusType'][samples]
        current_target = values['CurrentTarget'][samples]
        stimulus_code = values['StimulusCode'][samples]
        intervals = np.diff(samples) / raw_header.info['sfreq']
        target_values = current_target[current_target != 0]
        type_ok = bool(np.all(np.isin(stimulus_type, [0, 1])))
        target_constant = bool(len(target_values) > 0 and len(np.unique(target_values)) == 1)
        interval_ok = bool(len(intervals) == 0 or np.all(np.isclose(intervals, EXPECTED_INTERVAL_SEC, atol=1 / raw_header.info['sfreq'])))
        code_pattern = classify_code_sequence(stimulus_code)
        standard_ok = len(samples) == EXPECTED_FLASHES and int(np.sum(stimulus_type == 1)) == EXPECTED_TARGET and int(np.sum(stimulus_type == 0)) == EXPECTED_NON_TARGET
        structural_ok = type_ok and target_constant and interval_ok and code_pattern == 'monotonic_within_character'
        if not structural_ok:
            status = 'needs_review'
        elif fixed_standard and standard_ok:
            status = 'complete'
        elif fixed_standard:
            status = 'partial'
        else:
            status = 'needs_review'
        row = {**base, 'character_trial': character_index, 'n_flashes': len(samples), 'n_target': int(np.sum(stimulus_type == 1)), 'n_non_target': int(np.sum(stimulus_type == 0)), 'current_target_values': np.unique(target_values).tolist(), 'current_target_constant': target_constant, 'stimulus_type_values': np.unique(stimulus_type).tolist(), 'stimulus_type_only_0_1': type_ok, 'stimulus_code_start': float(stimulus_code[0]) if len(stimulus_code) else np.nan, 'stimulus_code_end': float(stimulus_code[-1]) if len(stimulus_code) else np.nan, 'stimulus_code_pattern': code_pattern, 'median_flash_interval_sec': float(np.median(intervals)) if len(intervals) else np.nan, 'interval_0_25_sec_ok': interval_ok, 'fixed_standard_applicable': fixed_standard, 'standard_counts_ok': standard_ok, 'character_status': status}
        character_qc_rows.append(row)
        rows_for_file.append(row)
        previous_boundary = boundary
    statuses = [row['character_status'] for row in rows_for_file]
    if not statuses:
        file_status, notes = 'needs_review', 'no character boundaries detected'
    elif any(status == 'needs_review' for status in statuses):
        file_status, notes = 'needs_review', 'structural issue or dynamic condition requires review'
    elif all(status == 'complete' for status in statuses):
        file_status, notes = 'complete', ''
    else:
        file_status, notes = 'partial', 'one or more characters fail the fixed count standard'
    file_qc_rows.append({**base, 'n_character_trials': len(rows_for_file), 'file_status': file_status, 'review_notes': notes})

character_event_qc = pd.DataFrame(character_qc_rows)
file_event_qc = pd.DataFrame(file_qc_rows)
display(character_event_qc)
display(file_event_qc)
display(file_event_qc.groupby(['split', 'condition', 'file_status']).size().reset_index(name='n_files'))
character_qc_export = PROJECT_ROOT / 'outputs' / 'raw_inspection' / 'StudyF_character_event_qc.csv'
file_qc_export = PROJECT_ROOT / 'outputs' / 'raw_inspection' / 'StudyF_file_event_qc.csv'
character_event_qc.to_csv(character_qc_export, index=False, encoding='utf-8-sig')
file_event_qc.to_csv(file_qc_export, index=False, encoding='utf-8-sig')
print('Saved:', character_qc_export)
print('Saved:', file_qc_export)

---

## 附录：edf_inspection_and_preprocessing

# Phase 1: bigP3BCI EDF Inspection and Initial Preprocessing

本阶段的目标是确认 EDF 可以读取，了解 EEG channel、sampling rate、annotations / stimulus markers、target/non-target 标签结构，并进行初步、可视化的信号质量检查。正式 preprocessing 参数将在确认 lab 代码后确定；当前仅使用论文中的探索性设置。

本 notebook 不训练 SWLDA、EEGNet 或任何分类模型。

## 1. Setup

导入依赖，定位项目根目录，并准备报告输出目录。此处不下载或复制 EDF 文件。

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne

print(f'MNE version: {mne.__version__}')

def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    start = start.resolve()
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / 'README.md').exists() or (candidate / '.git').exists():
            return candidate
    raise FileNotFoundError(
        f'Could not find project root from {start}. Start Jupyter inside the repository.'
    )
PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
REPORT_DIR = PROJECT_ROOT / 'outputs' / 'reports'
RAW_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Project root: {PROJECT_ROOT}')
print(f'Raw data directory: {RAW_DIR}')
print(f'Reports directory: {REPORT_DIR}')

## 2. Select one participant and EDF files

先只使用一位 Study L ALS 用户的一份 Train EDF 和匹配的 Test EDF。请填写下面三个参数；路径可以是绝对路径，也可以是相对于项目根目录的路径。

In [ ]:
SUBJECT_ID = "..."
TRAIN_EDF = Path("...")
TEST_EDF = Path("...")

def resolve_input_path(path):
    path = Path(path)
    return path if path.is_absolute() else PROJECT_ROOT / path

train_path = resolve_input_path(TRAIN_EDF)
test_path = resolve_input_path(TEST_EDF)
missing = [str(p) for p in (train_path, test_path) if not p.is_file()]
if SUBJECT_ID == '...' or '...' in (str(TRAIN_EDF), str(TEST_EDF)):
    print('请先填写 SUBJECT_ID、TRAIN_EDF 和 TEST_EDF，然后从本节重新运行。')
elif missing:
    print('以下 EDF 文件不存在，暂不继续读取：')
    for item in missing: print(f'  - {item}')
else:
    print(f'Subject: {SUBJECT_ID}')
    print(f'Train: {train_path}')
    print(f'Test:  {test_path}')

## 3. Read EDF files

仅在两个路径都有效时读取 EDF；不立即滤波，也不改变原始数据。

In [ ]:
if 'missing' not in globals() or missing or SUBJECT_ID == '...':
    print('跳过读取：请先完成上一节的 EDF 路径设置。')
else:
    raw_train = mne.io.read_raw_edf(train_path, preload=False, infer_types=True, verbose=False)
    raw_test = mne.io.read_raw_edf(test_path, preload=False, infer_types=True, verbose=False)
    def print_raw_summary(label, raw):
        print(f'\n{label}')
        print(f'  sampling rate: {raw.info["sfreq"]} Hz')
        print(f'  duration: {raw.n_times / raw.info["sfreq"]:.2f} s')
        print(f'  channels ({len(raw.ch_names)}): {raw.ch_names}')
        print(f'  inferred types: {dict(zip(raw.ch_names, raw.get_channel_types()))}')
        print(f'  annotations: {len(raw.annotations)}')
    print_raw_summary('Train EDF', raw_train)
    print_raw_summary('Test EDF', raw_test)

## 4. Channel inventory

下面分别列出 channel name、MNE 推断的 channel type 和可获取的 unit。论文中的 7 个 channel 仅作提示，不因大小写或命名差异自动判定失败。

In [ ]:
EXPECTED_EEG_CHANNELS = ['Fz', 'Cz', 'P3', 'Pz', 'P4', 'PO7', 'PO8']
def channel_inventory(raw):
    rows = []
    for name, kind, unit in zip(raw.ch_names, raw.get_channel_types(), raw._orig_units.values()):
        rows.append({'channel name': name, 'channel type': kind, 'unit': unit})
    return pd.DataFrame(rows)

if 'raw_train' in globals():
    train_inventory = channel_inventory(raw_train)
    test_inventory = channel_inventory(raw_test)
    display(train_inventory)
    display(test_inventory)
    for label, raw in [('Train', raw_train), ('Test', raw_test)]:
        present = [x for x in EXPECTED_EEG_CHANNELS if x in raw.ch_names]
        missing_expected = [x for x in EXPECTED_EEG_CHANNELS if x not in raw.ch_names]
        print(f'{label} expected channels present: {present}')
        print(f'{label} expected channels not exact-name matches: {missing_expected}')
        print('提示：请人工确认大小写、前缀或其他命名方式是否对应。')

## 5. Event and annotation inspection

先观察 annotations 的原始 description 和频数，不预设 event code。

**在确认 `StimulusType` 的实际存储方式和 target/non-target event code 之前，不进行 epoch 或标签映射。**

In [ ]:
def annotation_table(raw):
    return pd.DataFrame({'onset': raw.annotations.onset, 'duration': raw.annotations.duration, 'description': raw.annotations.description})
marker_words = ('stimulus', 'event', 'target', 'trigger', 'status', 'marker')
if 'raw_train' in globals():
    for label, raw in [('Train', raw_train), ('Test', raw_test)]:
        ann = annotation_table(raw)
        print(f'\n{label} annotations (first 20):')
        display(ann.head(20))
        if ann.empty:
            print('Annotations 为空；下一步需要检查 EDF 中的 event/stimulus channel。')
        else:
            print('Description counts:')
            display(ann['description'].value_counts())
            print('All unique descriptions:', ann['description'].unique().tolist())
        possible = [name for name in raw.ch_names if any(word in name.lower() for word in marker_words)]
        print(f'Possible marker/stimulus channels: {possible or "none found"}')

## 6. Raw EEG visualization

优先使用 7 个论文 channel 中实际存在的通道；否则使用 MNE 识别出的 EEG channels。每幅图显示前 10 秒原始波形。

In [ ]:
def choose_eeg_channels(raw):
    preferred = [x for x in EXPECTED_EEG_CHANNELS if x in raw.ch_names]
    return preferred or mne.pick_types(raw.info, eeg=True, exclude=[]).tolist()
def plot_raw_10s(label, raw):
    picks = choose_eeg_channels(raw)
    if not picks:
        print(f'{label}: 找不到 EEG channels，跳过波形图。')
        return
    data, times = raw.copy().pick(picks).get_data(start=0, stop=min(raw.n_times, int(10 * raw.info['sfreq'])), return_times=True)
    fig, axes = plt.subplots(len(picks), 1, sharex=True, figsize=(12, max(2, 1.5 * len(picks))))
    axes = np.atleast_1d(axes)
    for ax, signal, name in zip(axes, data, picks): ax.plot(times, signal); ax.set_ylabel(name); ax.grid(alpha=.3)
    axes[-1].set_xlabel('Time (s)'); fig.suptitle(f'{label} raw EEG: first 10 seconds'); fig.tight_layout(); plt.show()
if 'raw_train' in globals(): plot_raw_10s('Train', raw_train)
if 'raw_test' in globals(): plot_raw_10s('Test', raw_test)

## 7. Power spectral density (PSD) QC

PSD 范围为 0.5–80 Hz，用于观察频谱和可能的 60 Hz line noise。图片保存到 subject-specific report 目录。

In [ ]:
subject_report_dir = REPORT_DIR / str(SUBJECT_ID)
subject_report_dir.mkdir(parents=True, exist_ok=True) if SUBJECT_ID != '...' else None
def plot_psd(label, raw, filename):
    picks = choose_eeg_channels(raw)
    if not picks: print(f'{label}: 找不到 EEG channels，跳过 PSD。'); return
    spectrum = raw.compute_psd(fmin=0.5, fmax=80, picks=picks, reject_by_annotation=False, verbose=False)
    fig = spectrum.plot(average=True, show=False); fig.suptitle(f'{label} raw EEG PSD (0.5–80 Hz; inspect 60 Hz)')
    fig.savefig(subject_report_dir / filename, dpi=150, bbox_inches='tight'); plt.show(); plt.close(fig)
if 'raw_train' in globals(): plot_psd('Train', raw_train, 'train_raw_psd.png')
if 'raw_test' in globals(): plot_psd('Test', raw_test, 'test_raw_psd.png')

## 8. Exploratory filtering — do not overwrite raw data

以下是 Gupta et al. 2026 描述的探索性设置：60 Hz notch 和 0.5–30 Hz band-pass。reference、bad-channel / artifact handling 和精确参数仍待 lab 确认。过滤仅作用于副本，`raw_train` / `raw_test` 保持不变。

In [ ]:
def exploratory_filter(raw):
    filtered = raw.copy()
    picks = choose_eeg_channels(filtered)
    if not picks: return filtered, []
    filtered.notch_filter(freqs=[60], picks=picks, verbose=False)
    filtered.filter(l_freq=0.5, h_freq=30, picks=picks, verbose=False)
    return filtered, picks
if 'raw_train' in globals():
    raw_train_filtered, _ = exploratory_filter(raw_train)
    raw_test_filtered, _ = exploratory_filter(raw_test)
    plot_raw_10s('Train filtered', raw_train_filtered)
    plot_raw_10s('Test filtered', raw_test_filtered)
    plot_psd('Train filtered', raw_train_filtered, 'train_filtered_psd.png')
    plot_psd('Test filtered', raw_test_filtered, 'test_filtered_psd.png')

## 9. Phase-1 checklist

请人工确认：

- [ ] Train/Test EDF 都能读取
- [ ] sampling rate 是否为 256 Hz
- [ ] 7 个论文 EEG channels 是否存在
- [ ] `StimulusType` / target / non-target 的实际编码是什么
- [ ] event 位于 annotations 还是专门的 channel
- [ ] Train/Test 的 event structure 是否一致
- [ ] 是否存在显著 60 Hz noise、坏通道或异常记录

只有完成上述确认后，下一 notebook 才实现 stimulus-locked `0–800 ms` epochs 和 target/non-target 标签提取。

---

## 附录：bigp3bci_artifact_qc

# bigP3BCI Artifact QC：原始 EEG 初步检查

本 notebook 当前只完成三件事：查看原始 EEG 波形、查看 PSD、记录异常。
不修复数据、不运行 ICA、不删除 epoch、不应用统一滤波，也不覆盖原始 EDF。

当前检查代表性 Subject：`F_05`（Duke 推断组）、`F_03`（ETSU 推断组）、`F_25`（高频表现特殊）。
从本节开始，波形和 PSD 只使用论文中用于 P300 分类的 8 个 EEG 通道。
坏道/坏数据段处理将在下一部分单独进行。

In [ ]:
%matplotlib qt
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne

PROJECT_ROOT=Path.cwd()
while PROJECT_ROOT!=PROJECT_ROOT.parent and not ((PROJECT_ROOT/'README.md').exists() or (PROJECT_ROOT/'.git').exists()):
    PROJECT_ROOT=PROJECT_ROOT.parent
if not ((PROJECT_ROOT/'README.md').exists() or (PROJECT_ROOT/'.git').exists()):
    raise FileNotFoundError('请在项目目录或子目录启动 Jupyter。')

REPRESENTATIVE_SUBJECTS=['F_05','F_06','F_07','F_08','F_03','F_20','F_21','F_23','F_24','F_25']
SESSION_ID='SE001'
raw_train={}
for subject_id in REPRESENTATIVE_SUBJECTS:
    train_root=PROJECT_ROOT/'data'/'raw'/'bigP3BCI-data'/'StudyF'/subject_id/SESSION_ID/'Train'/'CBCol'
    files=sorted(train_root.glob('*.edf'))
    if not files:
        print('No EDF found:', subject_id, SESSION_ID)
        continue
    path=files[0]
    raw_train[f'{subject_id}_{SESSION_ID}_{path.name}']=mne.io.read_raw_edf(path,preload=False,infer_types=True,verbose=False)
print('MNE:',mne.__version__,'; representative files:',len(raw_train))
print(list(raw_train))


In [ ]:
import re

def assign_bigp3bci_channel_types(raw):
    event_names={
        'StimulusBegin','StimulusType','StimulusCode','CurrentTarget',
        'FakeFeedback','DisplayResults','DisplayResult','SelectedTarget',
        'SelectedRow','SelectedColumn','PhaseInSequence'
    }
    channel_types={}
    for name in raw.ch_names:
        if name in event_names:
            channel_types[name]='stim'
        elif re.match(r'^.+_\d+_\d+$', name):
            channel_types[name]='misc'
        elif name.upper().startswith('EOG'):
            channel_types[name]='eog'
    if channel_types:
        raw.set_channel_types(channel_types, on_unit_change='ignore', verbose=False)
    return raw

PAPER_EEG_CHANNELS=['EEG_Fz','EEG_Cz','EEG_P3','EEG_Pz','EEG_P4','EEG_PO7','EEG_PO8','EEG_Oz']

def get_signal_channels(raw):
    eeg_names=[name for name, kind in zip(raw.ch_names, raw.get_channel_types()) if kind == 'eeg']
    selected=[name for name in PAPER_EEG_CHANNELS if name in eeg_names]
    missing=[name for name in PAPER_EEG_CHANNELS if name not in eeg_names]
    if missing:
        print('Missing paper channels:', missing)
    return selected

raw_train={
    name: assign_bigp3bci_channel_types(raw)
    for name, raw in raw_train.items()
}
print('Selected paper EEG channels:', get_signal_channels(next(iter(raw_train.values()))))
print('Selected channel count:', len(get_signal_channels(next(iter(raw_train.values())))))

Duke_subjects=['F_05','F_06','F_07','F_08']
ETSU_subjects=['F_03','F_20','F_21','F_23','F_24','F_25']

def show_subject_group(group_name, subject_ids):
    print(f'===== {group_name} =====')
    for subject_id in subject_ids:
        matches=[(filename, raw) for filename, raw in raw_train.items() if filename.startswith(f'{subject_id}_')]
        if not matches:
            print('No loaded file:', subject_id)
            continue
        filename, raw = matches[0]
        eeg=get_signal_channels(raw)
        print(f'{subject_id}: {filename}; channels={len(eeg)}')
        raw.plot(duration=min(60,raw.times[-1]),start=0,picks=eeg,n_channels=len(eeg),remove_dc=False,scalings='auto',title=f'{subject_id}: raw EEG waveform',show=True)
        raw.compute_psd(fmin=0.5,fmax=40,picks=eeg,reject_by_annotation=False,verbose=False).plot(average=True,amplitude=False,show=True)
        raw.compute_psd(fmin=0.5,fmax=min(127,raw.info['sfreq']/2-1),picks=eeg,reject_by_annotation=False,verbose=False).plot(average=True,amplitude=False,show=True)


## 1. 数据与通道概览

先确认采样率、持续时间、通道类型和已有 bad-channel 标记。事件/反馈通道不参与 EEG 伪迹判断。


In [ ]:
overview=[]
for filename,raw in raw_train.items():
    kinds=pd.Series(raw.get_channel_types()).value_counts().to_dict()
    overview.append({'file':filename,'sfreq':raw.info['sfreq'],'duration_sec':raw.times[-1],'n_channels':len(raw.ch_names),'channel_types':kinds,'existing_bads':raw.info['bads']})
display(pd.DataFrame(overview))
for filename,raw in raw_train.items():
    print('\n',filename)
    display(pd.DataFrame({'channel':raw.ch_names,'type':raw.get_channel_types()}))


## 1. 查看原始 EEG 波形

使用 `Raw.plot()` 默认查看 60 秒原始 EEG，底部滑块仍可浏览完整记录。重点观察零通道、异常振幅、突然跳变、低频漂移和大幅伪迹。此步骤不滤波、不改变原始数据。


In [ ]:
show_subject_group('Duke (inferred)', Duke_subjects)


In [ ]:
show_subject_group('ETSU (inferred)', ETSU_subjects)

## 2. 查看 PSD

先看项目相关的 0.5–40 Hz，再看完整的 0.5–127 Hz。这里只显示频谱，不应用 notch 或其他滤波。


In [ ]:
show_subject_group('ETSU (inferred)', ETSU_subjects)


## 暂时跳过：EOG/眼动伪迹

当前阶段不运行 EOG detection；后续处理坏道和异常数据段后再决定。


In [ ]:
print('EOG detection skipped in this QC stage.')


## 暂时跳过：ECG/心跳伪迹

当前阶段不运行 ECG detection，也不从 EEG 强行推断 ECG。


In [ ]:
print('ECG detection skipped in this QC stage.')


## 后续步骤：Handling bad channels / bad data spans

本阶段暂不自动计算或标记 bad channels，也不处理异常连续区间。完成原始波形和 PSD 观察后，再进入这一部分。


In [ ]:
print('Bad-channel numerical screening deferred to the next stage.')


## 后续步骤：异常连续区间与 annotation

异常连续区间和 `BAD_` annotation 暂不处理，留到下一部分。


In [ ]:
print('Bad-span annotation handling deferred to the next stage.')


## 当前 notebook 的边界

本 notebook 只做 artifact QC。确认 QC 结果后，下一步才决定是否加入统一滤波、坏道处理、annotation、ICA 或 epoch rejection。跨 session 迁移实验中，所有处理参数都需要记录，并避免使用目标 session 的标签或测试结果泄漏到 source-session 训练流程。
